# Train Hierarchical BiLSTM + CRF on PubMed 200k
This notebook downloads `zj88zj/PubMed_200k_RCT` from Hugging Face, preprocesses data, and trains a Hierarchical BiLSTM + CRF model.

In [ ]:
%pip install -q datasets torchcrf scikit-learn tqdm

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchcrf import CRF
from sklearn.metrics import classification_report, f1_score
from tqdm.auto import tqdm

In [ ]:
# Resolve project root when running inside notebooks/
CWD = Path.cwd().resolve()
ROOT = CWD.parent if CWD.name == "notebooks" else CWD
PROCESSED_DIR = ROOT / "data" / "processed"

cmd = [
    sys.executable,
    str(ROOT / "preprocess_data.py"),
    "--dataset-name",
    "zj88zj/PubMed_200k_RCT",
    "--output-dir",
    str(PROCESSED_DIR),
    "--min-freq",
    "3",
    "--sent-percentile",
    "95",
    "--word-percentile",
    "95",
]

print("Running preprocess script...")
subprocess.run(cmd, check=True)
print("Done preprocess ->", PROCESSED_DIR)

In [ ]:
def read_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def read_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


metadata = read_json(PROCESSED_DIR / "metadata.json")
label2id = read_json(PROCESSED_DIR / "label2id.json")
id2label = {int(k): v for k, v in read_json(PROCESSED_DIR / "id2label.json").items()}

train_rows = read_jsonl(PROCESSED_DIR / "train.jsonl")
dev_rows = read_jsonl(PROCESSED_DIR / "dev.jsonl")
test_rows = read_jsonl(PROCESSED_DIR / "test.jsonl")

print(metadata)
print("train/dev/test sizes:", len(train_rows), len(dev_rows), len(test_rows))

In [ ]:
IGNORE_LABEL_ID = -100


class PubMedHierDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        return {
            "input_ids": torch.tensor(row["input_ids"], dtype=torch.long),
            "word_mask": torch.tensor(row["word_mask"], dtype=torch.bool),
            "sentence_mask": torch.tensor(row["sentence_mask"], dtype=torch.bool),
            "label_ids": torch.tensor(row["label_ids"], dtype=torch.long),
        }


def collate_batch(batch):
    return {
        "input_ids": torch.stack([x["input_ids"] for x in batch], dim=0),
        "word_mask": torch.stack([x["word_mask"] for x in batch], dim=0),
        "sentence_mask": torch.stack([x["sentence_mask"] for x in batch], dim=0),
        "label_ids": torch.stack([x["label_ids"] for x in batch], dim=0),
    }

In [ ]:
class HierBiLSTMCRF(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_labels: int,
        emb_dim: int = 200,
        word_hidden: int = 128,
        sent_hidden: int = 128,
        dropout: float = 0.5,
        pad_id: int = 0,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.word_lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=word_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.sent_lstm = nn.LSTM(
            input_size=word_hidden * 2,
            hidden_size=sent_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(sent_hidden * 2, num_labels)
        self.crf = CRF(num_tags=num_labels, batch_first=True)

    def _encode_sentences(
        self, input_ids: torch.Tensor, word_mask: torch.Tensor
    ) -> torch.Tensor:
        # input_ids: [B, S, W], word_mask: [B, S, W]
        bsz, max_s, max_w = input_ids.shape
        flat_input = input_ids.view(bsz * max_s, max_w)
        flat_mask = word_mask.view(bsz * max_s, max_w)

        emb = self.embedding(flat_input)
        lengths = flat_mask.sum(dim=1).clamp(min=1).cpu()

        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths=lengths, batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.word_lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=max_w
        )

        mask_float = flat_mask.unsqueeze(-1).float()
        sent_vec = (out * mask_float).sum(dim=1) / mask_float.sum(dim=1).clamp(min=1e-6)
        return sent_vec.view(bsz, max_s, -1)

    def _compute_emissions(
        self,
        input_ids: torch.Tensor,
        word_mask: torch.Tensor,
        sentence_mask: torch.Tensor,
    ):
        sent_repr = self._encode_sentences(input_ids, word_mask)

        sent_lengths = sentence_mask.sum(dim=1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(
            sent_repr, lengths=sent_lengths, batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.sent_lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=sent_repr.shape[1]
        )
        out = self.dropout(out)
        emissions = self.classifier(out)
        return emissions

    def neg_log_likelihood(self, input_ids, word_mask, sentence_mask, tags):
        emissions = self._compute_emissions(input_ids, word_mask, sentence_mask)
        nll = -self.crf(emissions, tags, mask=sentence_mask, reduction="mean")
        return nll

    def decode(self, input_ids, word_mask, sentence_mask):
        emissions = self._compute_emissions(input_ids, word_mask, sentence_mask)
        return self.crf.decode(emissions, mask=sentence_mask)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

BATCH_SIZE = 16
NUM_EPOCHS = 20
LR = 3e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 5

train_loader = DataLoader(
    PubMedHierDataset(train_rows),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_batch,
)
dev_loader = DataLoader(
    PubMedHierDataset(dev_rows),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_batch,
)
test_loader = DataLoader(
    PubMedHierDataset(test_rows),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_batch,
)

model = HierBiLSTMCRF(
    vocab_size=metadata["vocab_size"],
    num_labels=metadata["num_labels"],
    emb_dim=200,
    word_hidden=128,
    sent_hidden=128,
    dropout=0.5,
    pad_id=0,
).to(device)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2, verbose=True
)

In [ ]:
def to_device(batch, device):
    return {k: v.to(device) for k, v in batch.items()}


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc="Train", leave=False)

    for batch in pbar:
        batch = to_device(batch, device)

        optimizer.zero_grad(set_to_none=True)
        loss = model.neg_log_likelihood(
            batch["input_ids"],
            batch["word_mask"],
            batch["sentence_mask"],
            batch["label_ids"],
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / max(len(loader), 1)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_true = []
    all_pred = []

    for batch in tqdm(loader, desc="Eval", leave=False):
        batch = to_device(batch, device)
        preds = model.decode(
            batch["input_ids"], batch["word_mask"], batch["sentence_mask"]
        )

        true_tags = batch["label_ids"].tolist()
        mask = batch["sentence_mask"].tolist()

        for pred_seq, true_seq, m_seq in zip(preds, true_tags, mask):
            valid_len = sum(1 for m in m_seq if m)
            t = true_seq[:valid_len]
            p = pred_seq[:valid_len]
            all_true.extend(t)
            all_pred.extend(p)

    macro_f1 = f1_score(all_true, all_pred, average="macro")
    micro_f1 = f1_score(all_true, all_pred, average="micro")
    report = classification_report(
        all_true,
        all_pred,
        labels=sorted(id2label.keys()),
        target_names=[id2label[i] for i in sorted(id2label.keys())],
        digits=4,
        zero_division=0,
    )
    return macro_f1, micro_f1, report

In [ ]:
best_dev_f1 = -1.0
epochs_no_improve = 0
best_ckpt = ROOT / "models" / "best_model_pubmed200k_hf.pth"
best_ckpt.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    dev_macro_f1, dev_micro_f1, dev_report = evaluate(model, dev_loader, device)

    scheduler.step(dev_macro_f1)

    print(
        f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | dev_macro_f1={dev_macro_f1:.4f} | dev_micro_f1={dev_micro_f1:.4f}"
    )

    if dev_macro_f1 > best_dev_f1:
        best_dev_f1 = dev_macro_f1
        epochs_no_improve = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "label2id": label2id,
                "metadata": metadata,
            },
            best_ckpt,
        )
        print("Saved best model ->", best_ckpt)
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

In [ ]:
checkpoint = torch.load(best_ckpt, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

test_macro_f1, test_micro_f1, test_report = evaluate(model, test_loader, device)
print(f"Test macro F1: {test_macro_f1:.4f}")
print(f"Test micro F1: {test_micro_f1:.4f}")
print(test_report)